# Committor Model: 1D Underdamped Cosine Well, Staple Paths, Mirrored Pieces and Lag-Δ Fit

Trains `CommittorNE` on the staple run `sim_istarwell2rv` with the settings that worked on the 2D Brownian Z staple run (mirrored pieces, rejection weights) and on the well MD (a lag-Δ fit), to see whether they also fix this system.

**Why:** the earlier staple fit used forward pieces only and a lag-1 fit: RMS 0.019–0.031 against the exact q(x, v), `zqs_z` ≈ 3, with error bands along the energy contour through the bump tops (`docs/results_overview.md` II.2). On the 2D Brownian Z the mirrored pieces were required (notes §9.1); on the well MD a lag-16 fit removed the energy-contour bands (notes §10.1).

**Simulation:** `sim_istarwell2rv`: staple, cosine well (1.2 kT, two 0.15 kT bumps), Langevin γ = 1, kT = 1, dt = 0.005 (one frame per step), velocities recorded in `order.txt`. A: x ≤ −0.35, B: x ≥ 0.35.

**Exact committor:** brute-force q(x, v) on a 29 × 29 grid (`figures/well_check/reference.npz`, `reference_well.py`). The grid is coarse: interpolating it gives small errors that the stopped Z_q sees at short lags (section 3).

**Results so far** (`figures/well_check/run.log`, `run_rev_recorded.log`; RMS against exact over all frames with |v| < 3.5):

| Pieces | Velocity | Fit lag | RMS | max \|z\| at lags 1, 2, 4, 16, 64, 256 |
|---|---|---|---|---|
| forward | recorded | 1 | 0.019 | 0.5 0.8 0.9 1.6 2.3 2.7 |
| forward | recorded | 4 | 0.015 | 2.7 1.1 0.4 1.5 1.4 1.8 |
| forward | finite difference | 1 | 0.032 | 0.5 7.8 10.0 8.8 6.4 5.8 |
| forward | finite difference | 4 | 0.021 | 14.2 4.6 0.3 1.6 1.9 2.1 |
| forward + mirrored | recorded | 1 | 0.030 | 0.6 2.8 4.2 4.7 3.4 3.6 |
| forward + mirrored | recorded | 4 | 0.017 | 3.3 1.4 0.4 2.4 1.0 1.7 |
| forward + mirrored | finite difference | 1, 4 | not run yet | |

## 1. Imports

In [ ]:
import argparse
import os
import sys

import numpy as np
import matplotlib.pyplot as plt
import cupy as cp
from scipy.interpolate import RegularGridInterpolator

plt.rc('font', size=10)
plt.rc('axes', titlesize=14)
plt.rc('axes', labelsize=14)
plt.rc('xtick', labelsize=12)
plt.rc('ytick', labelsize=12)
plt.rc('legend', fontsize=12)

REPO = '/mnt/0bf0c339-34bb-4500-a5fb-f3c2a863de29/DATA/OP_improvement/optimalrcs'
sys.path.insert(0, f'{REPO}/optimalrcs-cupy/')
sys.path.insert(0, f'{REPO}/examples/')
import optimalrcs, metrics_tis as M
import tis_data
FIG = f'{REPO}/examples/figures'

# Figures of this notebook go to their own directory; the file names carry the run settings (TAG)
FIG_DIR = f'{REPO}/examples/figures/nb_well_staple'
os.makedirs(FIG_DIR, exist_ok=True)
def save_fig(name):
    plt.gcf().savefig(f'{FIG_DIR}/{name}.png', dpi=120, bbox_inches='tight')

## 2. Data Loading

`tis_data.prepare_data` cuts every accepted path at its first frame past M, adds the mirrored piece (the part before the last M crossing, read backwards, with the recorded velocity negated), and gives each frame its path's rejection weight. `--fd-velocity` adds a finite-difference velocity (change of x over one frame, computed on the whole path before the cut; in mirrored pieces it is computed backwards, so it is the velocity of the reversed dynamics).

**Parameters:**
| Parameter | Value | Description |
|-----------|-------|-------------|
| `PATHS_PER_ENSEMBLE` | 2000 | Random subset of accepted paths per ensemble, weights kept |
| `REVERSED` | True | Add the mirrored pieces (False: forward pieces only, as the earlier fit) |
| `WEIGHTS` | `rejections` | MC path weights |

Columns of `data.X`: x, recorded v, finite-difference Δx per frame.

In [ ]:
TIS_DIR = '/mnt/0bf0c339-34bb-4500-a5fb-f3c2a863de29/DATA/PyRETIS3/toytis/simulations/sim_istarwell2rv'
PATHS_PER_ENSEMBLE = 2000
SEED = 0
REVERSED = True
WEIGHTS = 'rejections'
DT = 0.005                         # MD time step = time per frame

parser = argparse.ArgumentParser()
tis_data.add_data_arguments(parser)
argv = [TIS_DIR, '--weights', WEIGHTS, '--paths-per-ensemble', str(PATHS_PER_ENSEMBLE), '--seed', str(SEED),
        '--fd-velocity']
if REVERSED:
    argv += ['--reversed']
data = tis_data.prepare_data(parser.parse_args(argv))

x, v_rec, v_fd = data.X[:, 0], data.X[:, 1], data.X[:, 2] / DT
b = data.boundary0 | data.boundary1
tau = M.stop_index(data.i_traj, b)             # first A/B frame or the piece's last frame
print(f'{len(x)} frames in {len(np.unique(data.i_traj))} pieces from {len(np.unique(data.group_traj))} paths')
print(f'corr(recorded v, fd v) {np.corrcoef(v_rec, v_fd)[0, 1]:.4f}, std of the difference {np.std(v_fd - v_rec):.3f}')

## 3. Data Test with the Exact Committor

Before any fit: put the exact committor into the stopped Z_q of the data. For unbiased data it gives |z| ≲ 2 at every lag; if it fails, no fit on these data can be right (notes §9.1).

On this run the exact committor fails at short lags with and without the mirrored pieces (forward only: 12.0 11.1 9.6 3.5 2.5 2.1; with mirrored pieces: 15.4 14.2 12.2 5.2 2.4 2.2). The failure grows by about √2 with twice the frames, as a fixed error would, and is largest at lag 1, where small interpolation errors of the coarse 29 × 29 reference grid matter most. So it points at the reference, not at a bias of the data. At lags ≥ 64 the data pass either way.

In [ ]:
ref = np.load(f'{FIG}/well_check/reference.npz')
interp = RegularGridInterpolator((ref['xs'], ref['vs']), ref['q'], bounds_error=False, fill_value=None)

def q_exact(vel):
    q = np.clip(interp(np.c_[np.clip(x, ref['xs'][0], ref['xs'][-1]), np.clip(vel, ref['vs'][0], ref['vs'][-1])]), 0, 1)
    q[data.boundary0] = 0; q[data.boundary1] = 1
    return q

LAGS = (1, 2, 4, 16, 64, 256)
def z_per_lag(r):
    """Worst |stopped Z_q / standard error| over q, per lag."""
    out = []
    for dt in LAGS:
        _, z = M.comp_zq_stopped_zscore(cp.asarray(r), b, data.i_traj, data.path_weights, dt, tau=tau,
                                        g_traj=data.group_traj)
        out.append(float(cp.max(cp.abs(z[5:]))))
    return ' '.join(f'{z:5.1f}' for z in out)

q_ref = q_exact(v_rec)                         # the truth uses the true (recorded) velocity
print(f'exact q, recorded v : max|z| at lags {LAGS}: {z_per_lag(q_ref)}')
print(f'exact q, fd v       : max|z| at lags {LAGS}: {z_per_lag(q_exact(v_fd))}')

## 4. Train Committor (Non-Parametric), Lag-Δ Fit

CVs (x, v) with v the recorded or the finite-difference velocity; each iteration draws one of them as the CV `y`.

**Lag-Δ fit** (notes §10.1): the library's update solves Σ_t f_k(t)[q(t+1) − q(t)] = 0 over one step. Here it is replaced by Σ_t w_t f_k(t)[q(min(t+Δ, τ_t)) − q(t)] = 0, with τ_t the first A/B frame or the last frame of the piece: windows stop at A/B or at the piece end, as in the stopped Z_q. Over one step underdamped dynamics nearly conserves energy, so errors in q that depend only on energy are almost invisible at lag 1; at lag Δ they become ~Δ times more visible. Rule of thumb γ·Δ·dt ≈ 0.05–0.1 (here Δ = 4 gives 0.02; on the MD Δ = 16 worked best). `FIT_LAG = 1` gives the library's lag-1 update. Judge a lag-Δ fit at lags ≥ Δ.

**Parameters** (as `figures/well_check/well_test.py`):
| Parameter | Value | Description |
|-----------|-------|-------------|
| `VELOCITY` | `recorded` | `recorded` or `fd` (positions only) |
| `FIT_LAG` | 4 | Fit lag Δ in frames |
| `envelope_scale` | 0.01 | RC envelope width (library default) |
| `gamma` | 0.1 | Regularization (scales the step) |
| `ny` | 6 | Polynomial degree |
| `max_iter` | 4000 | Iterations |

In [ ]:
VELOCITY = 'recorded'
FIT_LAG = 4
MAX_ITER = 4000
TAG = f"{'rev' if REVERSED else 'fwd'}_{VELOCITY}_lag{FIT_LAG}"   # run settings, in the figure names
optimalrcs.envelope_scale = 0.01

vel = v_rec if VELOCITY == 'recorded' else v_fd
Y = cp.asarray(np.c_[x, vel])
W = cp.asarray(data.path_weights)

if not hasattr(optimalrcs.nonparametrics, 'npneq_lag1'):
    optimalrcs.nonparametrics.npneq_lag1 = optimalrcs.nonparametrics.npneq     # keep the library's update
if FIT_LAG == 1:
    optimalrcs.nonparametrics.npneq = optimalrcs.nonparametrics.npneq_lag1
else:
    n = len(x); tau_np = cp.asnumpy(tau)
    u_np = np.minimum(np.arange(n) + FIT_LAG, tau_np)
    U = cp.asarray(u_np)
    STARTS = cp.asarray(np.flatnonzero(~b & (u_np > np.arange(n))))

    def npneq_lag(r_traj, fk, i_traj=None, gamma=0, stable=False, train_mask=None, chunk=None, w_traj=None, **kw):
        akj, bb = 0, 0
        for s0 in range(0, STARTS.size, 500_000):
            st = STARTS[s0:s0 + 500_000]
            fa, fb, wt = fk[:, st], fk[:, U[st]], W[st]
            akj = akj + fa @ (fb * wt - fa * (wt + gamma)).T
            bb = bb + fa @ (-(r_traj[U[st]] - r_traj[st]) * wt)
        al = cp.linalg.lstsq(akj, bb, rcond=None)[0]
        return cp.clip(r_traj + al @ fk, 0, 1)
    optimalrcs.nonparametrics.npneq = npneq_lag

np.random.seed(0); cp.random.seed(0)
q = optimalrcs.CommittorNE(
    boundary0=data.boundary0, boundary1=data.boundary1,
    i_traj=data.i_traj,            # piece (forward or mirrored)
    t_traj=data.t_traj,
    path_weights=data.path_weights,
    group_traj=data.group_traj,    # path: its two pieces are not independent
)
q.set_fixed_traj_length_trap(b, 2 ** 16)       # A/B ends as traps, for the metrics only

q.fit_transform(
    lambda: Y[:, np.random.randint(2)],
    basis_functions=lambda r_, y_, n_, f_: optimalrcs.basis_poly_ry(r_, y_, n_, f_),
    gamma=0.1,
    ny=6,
    max_iter=MAX_ITER,
    print_step=500,
    metrics_print=('iter', 'delta_r2', 'max_z_zq_stopped', 'delta_x', 'time_elapsed'),
    save_min_delta_zq=False,
)
q.plots_metrics(['delta_r2', 'max_z_zq_stopped', 'delta_x'])
save_fig(f'metrics_{TAG}')
plt.show()
q.plots_tis()
save_fig(f'tis_diagnostics_{TAG}')

## 5. Comparison with the Exact Committor

RMS of fit − exact over all frames with |v| < 3.5, the mean error near A and near B for slow particles (|x| > 0.25, |v| < 0.5), and the worst stopped z per lag. Compare with the table at the top.

In [ ]:
r = cp.asnumpy(q.r_traj)
inside = ~b & (np.abs(v_rec) < 3.5)
near_a = inside & (x < -0.25) & (np.abs(v_rec) < 0.5)
near_b = inside & (x > 0.25) & (np.abs(v_rec) < 0.5)
err = r - q_ref
print(f'{"forward + mirrored" if REVERSED else "forward"}, {VELOCITY} v, lag {FIT_LAG}: '
      f'RMS {np.sqrt(np.mean(err[inside] ** 2)):.3f} | near A {err[near_a].mean():+.3f} | near B {err[near_b].mean():+.3f}')
print(f'max|z| at lags {LAGS}: {z_per_lag(r)}')

e_x, e_v = np.linspace(-0.35, 0.35, 36), np.linspace(-3.5, 3.5, 36)
n_b = np.histogram2d(x[inside], v_rec[inside], [e_x, e_v])[0]
bm = lambda val: np.where(n_b >= 20, np.histogram2d(x[inside], v_rec[inside], [e_x, e_v], weights=val[inside])[0]
                          / np.maximum(n_b, 1), np.nan)
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for ax, (m, title, cmap, lim) in zip(axes, [(bm(r), 'fit', 'coolwarm', (0, 1)), (bm(q_ref), 'exact', 'coolwarm', (0, 1)),
                                           (bm(err), 'fit - exact', 'PuOr', (-0.15, 0.15))]):
    im = ax.pcolormesh(e_x, e_v, m.T, cmap=cmap, vmin=lim[0], vmax=lim[1])
    ax.set(title=title, xlabel='x', ylabel='v (recorded)')
    fig.colorbar(im, ax=ax)
plt.tight_layout()
save_fig(f'q_x_v_{TAG}')
plt.show()

## 6. Save Results

Uncomment to save the fit.

In [ ]:
# tag = f'{"rev_" if REVERSED else ""}{VELOCITY}_lag{FIT_LAG}'
# np.savez(f'{FIG}/well_check/fit_{tag}.npz', r=r)